# BƯỚC 4: Xây dựng Module Phân loại - Machine Learning
## ML Classification - Train Decision Tree, KNN, SVM

Huấn luyện các mô hình ML để phân loại tế bào dựa trên đặc trưng
- So sánh hiệu suất các mô hình
- Chọn mô hình tốt nhất
- Lưu model cho inference

## 1. Setup & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, recall_score, precision_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path.cwd()
print(f"Working directory: {BASE_DIR}")
print("✓ All libraries imported!")

## 2. Load Features CSV

In [ ]:
print("\n" + "="*60)
print("📥 LOADING FEATURES")
print("="*60)

features_path = BASE_DIR / 'data/processed/features/train_features.csv'

if features_path.exists():
    df = pd.read_csv(features_path)
    print(f"\n✓ Loaded features from: {features_path}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns[:5])}...")
else:
    print(f"⚠️ Features file not found at {features_path}")
    print("   → Run Bước 3: Feature Extraction first")

## 3. Data Exploration & Preprocessing

In [ ]:
print("\n" + "="*60)
print("🔍 DATA EXPLORATION")
print("="*60)

print(f"\nDataset info:")
print(df.info())

print(f"\nMissing values:")
print(df.isnull().sum().sum())

print(f"\nFirst rows:")
print(df.head())

# Statistics
print(f"\nBasic statistics:")
print(df.describe())

## 4. Prepare Data for Training

In [ ]:
print("\n" + "="*60)
print("🔧 PREPARING DATA")
print("="*60)

# Remove non-feature columns
feature_cols = [col for col in df.columns if col not in ['image_name', 'label']]
X = df[feature_cols]
y = df.get('label', pd.Series(np.zeros(len(df))))  # Dummy label if not present

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Handle missing values
X = X.fillna(X.mean())
print(f"✓ Missing values handled")

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Features standardized")

# Save scaler
scaler_path = BASE_DIR / 'models/ml/scaler.pkl'
scaler_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, scaler_path)
print(f"✓ Scaler saved")

## 5. Train Multiple ML Models

In [ ]:
print("\n" + "="*60)
print("🚀 TRAINING ML MODELS")
print("="*60)

models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, min_samples_split=5, random_state=42
    ),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
    'SVM (Linear)': SVC(kernel='linear', C=1.0, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
}

training_results = {}

for name, model in models.items():
    print(f"\n▶ Training {name}...")
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    training_results[name] = {
        'model': model,
        'accuracy': acc,
        'f1_score': f1,
        'y_pred': y_pred
    }
    
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1 Score: {f1:.4f}")

## 6. Model Comparison & Evaluation

In [ ]:
print("\n" + "="*60)
print("📊 MODEL COMPARISON")
print("="*60)

# Summary table
results_df = pd.DataFrame({
    'Model': list(training_results.keys()),
    'Accuracy': [v['accuracy'] for v in training_results.values()],
    'F1 Score': [v['f1_score'] for v in training_results.values()]
})

results_df = results_df.sort_values('Accuracy', ascending=False)
print("\n" + results_df.to_string(index=False))

# Best model
best_model_name = results_df.iloc[0]['Model']
best_model = training_results[best_model_name]['model']
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")

## 7. Detailed Evaluation of Best Model

print("\n" + "="*60)
print(f"📈 BEST MODEL PERFORMANCE: {best_model_name}")
print("="*60)

y_pred_best = training_results[best_model_name]['y_pred']

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
print(f"\nConfusion Matrix:")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 8. Save Best Model

print("\n" + "="*60)
print("💾 SAVING BEST MODEL")
print("="*60)

model_output_path = BASE_DIR / 'models/ml/best_model.pkl'
model_output_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, model_output_path)
print(f"\n✓ Best model saved to: {model_output_path}")
print(f"  Model type: {type(best_model).__name__}")
print(f"  File size: {model_output_path.stat().st_size / 1024:.2f} KB")

## 9. Cross-validation

print("\n" + "="*60)
print("🔄 CROSS-VALIDATION")
print("="*60)

cv_scores = cross_val_score(
    best_model, X_train_scaled, y_train, cv=5, scoring='accuracy'
)

print(f"\nCross-validation scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 10. Feature Importance

if hasattr(best_model, 'feature_importances_'):
    print("\n" + "="*60)
    print("📊 FEATURE IMPORTANCE")
    print("="*60)
    
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[-10:][::-1]  # Top 10
    
    print("\nTop 10 Important Features:")
    for i, idx in enumerate(indices):
        print(f"{i+1}. {feature_cols[idx]}: {importances[idx]:.4f}")
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.title('Top 10 Feature Importance')
    plt.bar(range(len(indices)), importances[indices])
    plt.xticks(range(len(indices)), [feature_cols[i] for i in indices], rotation=45)
    plt.tight_layout()
    plt.show()

## 12. Tóm tắt Bước 4